In [322]:
import pandas as pd
import pickle

df  = pd.read_parquet("data.parquet")

X_test  = pd.read_parquet("X_test.parquet")
y_test  = pd.read_parquet("y_test.parquet")

# dans l'approche LLM, il n'est pas possible de prédire un Tag sans le texte de réclamation
filter = (X_test["Consumer Claim"].isna() == False)
X_test  = X_test[filter]
y_test  = y_test[filter]

# Rechargement
with open("categories.pkl", "rb") as f:
    categories = pickle.load(f)

# Approche LLM

On demande au LLM de choisir parmis la catégorie 'Tag' en fonction de la demande client 'Consumer Claim'

In [323]:
import os
import pandas as pd

from mistralai.client import Mistral


# Client Mistral
client = Mistral(
    api_key=os.environ["MISTRAL_API_KEY"]
)


# ---------------------------------------------------------
# Préparation du prompt système
# ---------------------------------------------------------

SYSTEM_PROMPT = """
Vous êtes un système de classification de réclamations clientes.

Votre tâche consiste à attribuer à chaque réclamation UNE SEULE catégorie parmi les catégories autorisées.

Vous devez retourner exactement le nom d'une catégorie présente dans la liste fournie, sans explication supplémentaire.

Catégories autorisées :
{categories}

{additional_prompt}
"""


# ---------------------------------------------------------
# Fonction de classification
# ---------------------------------------------------------

def classify_with_llm(
    claim: str,
    categories: list[str],
    model: str = "mistral-small-latest",
    temperature: float = 0.0,
    additional_prompt: str = None
) -> str:
    """
    Classifie une réclamation client à l'aide d'un modèle de langage Mistral.

    La fonction construit un prompt système contenant la liste des catégories
    autorisées, puis soumet la réclamation au modèle afin qu'il détermine la
    catégorie correspondante.

    Args:
        claim (str):
            Texte de la réclamation client à classifier.

        categories (list[str]):
            Liste des catégories autorisées pour la classification.
            Le modèle doit retourner exactement l'une de ces catégories.

        model (str, optional):
            Identifiant du modèle Mistral utilisé pour la classification.
            Par défaut, "mistral-small-latest".

        temperature (float, optional):
            Paramètre contrôlant le niveau de variabilité de la réponse.
            Une valeur de 0.0 est utilisée par défaut afin de rendre la
            classification aussi déterministe que possible.

    Returns:
        str:
            Catégorie prédite par le modèle. Les espaces superflus au début
            et à la fin de la réponse sont supprimés.

    Raises:
        Exception:
            Une exception peut être levée si l'appel à l'API Mistral échoue
            ou si la réponse retournée ne possède pas le format attendu.

    Example:
        categories = [
            "Checking or savings account",
            "Debt collection",
            "Mortgage",
        ]

        category = classify_with_llm(
            claim="I have a problem with my mortgage payment.",
            categories=categories,
        )

        print(category)
    """

    system_prompt = SYSTEM_PROMPT.format(
        categories="\n".join(f"- {category}" for category in categories),
        additional_prompt=additional_prompt
    )

    user_prompt = f"""
Réclamation à classifier :

{claim}

Retournez uniquement la catégorie correspondante.
"""

    response = client.chat.complete(
        model=model,
        temperature=temperature,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
    )

    return response.choices[0].message.content.strip()

In [324]:
import random
import time

def with_retry(func: callable, max_retries=5):
    """
    Exécute une fonction avec une stratégie de nouvelle tentative automatique.

    La fonction est exécutée et, en cas d'exception, elle est relancée
    automatiquement jusqu'à atteindre le nombre maximal de tentatives.
    Le délai entre les tentatives augmente de manière exponentielle et
    une durée aléatoire est ajoutée afin d'éviter que plusieurs appels
    simultanés ne soient effectués exactement au même moment.

    Lorsque le nombre maximal de tentatives est atteint, l'utilisateur
    peut choisir de poursuivre le traitement en réinitialisant le compteur
    de tentatives ou d'arrêter le traitement.

    Args:
        func (callable):
            Fonction sans argument à exécuter.

        max_retries (int, optional):
            Nombre maximal de tentatives consécutives avant de demander
            à l'utilisateur s'il souhaite poursuivre. Par défaut à 5.

    Returns:
        Any:
            Résultat retourné par `func` lorsque son exécution réussit.

        None:
            Si le nombre maximal de tentatives est atteint et que
            l'utilisateur choisit d'arrêter le traitement.

    Raises:
        Exception:
            Les exceptions levées par `func` sont interceptées. Elles ne
            sont donc pas propagées lorsque l'utilisateur choisit de
            poursuivre ou d'arrêter le traitement.

    Notes:
        Le délai avant chaque nouvelle tentative suit la formule :

            2 ** attempt + random.uniform(0, 1)

        Il augmente donc progressivement afin de limiter les appels
        répétés et rapprochés vers le service distant.

    Example:
        result = with_retry(
            lambda: client.chat.complete(
                model="mistral-small-latest",
                messages=messages
            )
        )
    """
    attempt = 0

    while True:
        try:
            return func()

        except Exception as e:
            attempt += 1

            if attempt >= max_retries:
                print(f"Échec de l'appel : {e}")

                answer = input(
                    "Le nombre de tentatives est écoulé. "
                    "Continuer tout de même ? (o/n) : "
                ).strip().lower()

                if answer not in ["o", "oui", "y", "yes"]:
                    print("Traitement arrêté.")
                    return None

                # Nouvelle série de tentatives
                attempt = 0
                continue

            wait_time = (
                2 ** attempt
                + random.uniform(0, 1)
            )

            print(f"Erreur de l'appel : {e}")
            print(
                f"Nouvelle tentative dans "
                f"{wait_time:.2f} secondes..."
            )

            time.sleep(wait_time)

In [325]:
X_test.head(10)

,Consumer Claim,Company,Date received,Submitted via,Tags,State
192075,I generally let people walk over me you could ...,JPMORGAN CHASE & CO.,2018-07-23,Web,NaN,FL
131727,MR. XXXX calls and tells me he is with the leg...,Critical Resolution Mediation LLC,2018-10-19,Web,NaN,TX
455266,I am including my marriage license per your re...,"EQUIFAX, INC.",2017-07-25,Web,NaN,TN
80575,Disputed with company on XX/XX/XXXX. The compa...,WELLS FARGO & COMPANY,2019-01-09,Web,NaN,CA
551360,"On XXXX XXXX, XXXX, we turned-over our XXXX XX...","SUNTRUST BANKS, INC.",2017-02-22,Web,Older American,GA
694534,Transunion deleted XXXX XXXX and XXXX. I have ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",2016-06-15,Web,Older American,FL
776888,I contacted CFPB two years ago about how Bayvi...,"BAYVIEW LOAN SERVICING, LLC",2016-01-04,Web,NaN,GA
192053,We have received multiple calls from Commerica...,Commercial Acceptance Company,2018-07-23,Web,NaN,PA
721113,XXXX alleged that I owe them {$76.00}. for a p...,"Southwest Credit Systems, L.P.",2016-04-24,Web,Older American,NY
49478,To whom it may concern On XX/XX/XXXX Radius Gl...,Radius Global Solutions LLC,2019-02-26,Web,NaN,FL


# Fonctions de stockage des résultats

In [326]:
import pickle
from pathlib import Path

def load_results(results_path):
    if results_path.exists():
        # sauvegarde également les différentes catègories
        with open(results_path, "rb") as f:
            return pickle.load(f)
    else:
        return pd.DataFrame()
        
def save_results(results, results_path):
    # sauvegarde également les différentes catègories
    with open(results_path, "wb") as f:
        pickle.dump(results, f)

# Fonction de test

In [327]:
import time
import pandas as pd


def test(indices, additional_prompt: str = None) -> pd.DataFrame:
    """
    Évalue les performances du modèle de classification sur un ensemble
    d'indices du jeu de test.

    Pour chaque réclamation, la classification est effectuée via
    `with_retry`. Une exception est levée si aucune réponse valide
    n'est obtenue.

    Args:
        indices:
            Collection d'indices correspondant aux lignes de `X_test` et
            `y_test` à utiliser pour l'évaluation.

    Returns:
        pd.DataFrame:
            Tableau récapitulatif contenant les résultats de chaque test.

    Raises:
        RuntimeError:
            Si `with_retry` retourne `None`, indiquant qu'aucune réponse
            n'a pu être obtenue auprès du modèle.
    """

    results = []

    for idx in indices:
        X = X_test.loc[idx]
        y = y_test.loc[idx]

        question = X["Consumer Claim"]
        expected = y["Tag"]

        start = time.perf_counter()

        response = with_retry(
            lambda: classify_with_llm(
                question,
                categories,
                additional_prompt=additional_prompt
            )
        )

        elapsed = time.perf_counter() - start

        if response is None:
            break
        
        results.append({
            "Index": idx,
            "Question": question,
            "Réponse": response,
            "Attendue": expected,
            "Correct": response == expected,
            "Temps (s)": elapsed
        })

    return pd.DataFrame(results)

# Test avec 1 ligne de données du dataset

In [328]:
from metrics import (
    calculate_metrics,
    print_metrics,
    report,
)

name = "test 1"
results_path = Path(name + ".pkl")
results = load_results(results_path)

if len(results) == 0:
    print("Démarre le test:", name)
    indices = X_test.index[:1]
    results = test(indices)
    save_results(results, results_path)

metrics = calculate_metrics(results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
report(results)


------------------------
test 1
------------------------
Accuracy : 100.00%
Precision (macro) : 100.00%
Recall (macro) : 100.00%
F1 (macro) : 100.00%
Precision (weighted) : 100.00%
Recall (weighted) : 100.00%
F1 (weighted) : 100.00%
Temps moyen (s) : 0.440 s
Temps médian (s) : 0.440 s
Temps P95 (s) : 0.440 s
------------------------
                                                                              precision    recall  f1-score   support

Credit reporting, credit repair services, or other personal consumer reports       1.00      1.00      1.00         1

                                                                    accuracy                           1.00         1
                                                                   macro avg       1.00      1.00      1.00         1
                                                                weighted avg       1.00      1.00      1.00         1



# Test avec 20 lignes de données du dataset

In [329]:
from metrics import (
    calculate_metrics,
    print_metrics,
    report,
)

name = "test 20"
results_path = Path(name + ".pkl")
results = load_results(results_path)

if len(results) == 0:
    print("Démarre le test:", name)
    indices = X_test.index[:20]
    results = test(indices)
    save_results(results, results_path)

metrics = calculate_metrics(results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
report(results)


------------------------
test 20
------------------------
Accuracy : 50.00%
Precision (macro) : 19.44%
Recall (macro) : 25.00%
F1 (macro) : 21.30%
Precision (weighted) : 47.50%
Recall (weighted) : 50.00%
F1 (weighted) : 48.33%
Temps moyen (s) : 0.584 s
Temps médian (s) : 0.484 s
Temps P95 (s) : 1.015 s
------------------------
                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       0.00      0.00      0.00         2
                                                 Credit card or prepaid card       0.50      1.00      0.67         1
Credit reporting, credit repair services, or other personal consumer reports       0.50      0.50      0.50         6
                                                             Debt collection       0.75      0.75      0.75         8
                                                                    Mortgage   

## Test avec un prompt plus détaillé

In [330]:

# ---------------------------------------------------------
# Préparation du prompt système
# ---------------------------------------------------------

additional_prompt = """
Pour effectuer la classification :
1. Analysez le problème principal décrit dans la réclamation.
2. Identifiez le produit ou service financier concerné.
3. Comparez le problème avec les définitions des catégories disponibles.
4. Sélectionnez la catégorie qui correspond le mieux au problème principal.
5. Ne sélectionnez jamais une catégorie uniquement parce qu'un mot de la réclamation lui est associé.
"""


In [331]:
from metrics import (
    calculate_metrics,
    print_metrics,
    report,
)

name = "test 20 avec prompt plus détaillé"
results_path = Path(name + ".pkl")
results = load_results(results_path)

if len(results) == 0:
    print("Démarre le test:", name)
    indices = X_test.index[:20]
    results = test(indices)
    save_results(results, results_path)

metrics = calculate_metrics(results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
report(results)


------------------------
test 20 avec prompt plus détaillé
------------------------
Accuracy : 50.00%
Precision (macro) : 19.44%
Recall (macro) : 25.00%
F1 (macro) : 21.30%
Precision (weighted) : 47.50%
Recall (weighted) : 50.00%
F1 (weighted) : 48.33%
Temps moyen (s) : 0.444 s
Temps médian (s) : 0.421 s
Temps P95 (s) : 0.671 s
------------------------
                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       0.00      0.00      0.00         2
                                                 Credit card or prepaid card       0.50      1.00      0.67         1
Credit reporting, credit repair services, or other personal consumer reports       0.50      0.50      0.50         6
                                                             Debt collection       0.75      0.75      0.75         8
                                                     

## Test avec des exemples pour chaque catégorie

In [332]:
examples = (
    df.dropna(subset=["Consumer Claim"])
      .groupby("Tag", group_keys=False)
      .sample(n=3, random_state=42)
      .sort_values("Tag")
)

examples_text = "\n\n".join(
    f"Catégorie : {row['Tag']}\n"
    f"Réclamation : {row['Consumer Claim']}"
    for _, row in examples.iterrows()
)

In [333]:

# ---------------------------------------------------------
# Préparation du prompt système
# ---------------------------------------------------------

additional_prompt = """
Vous êtes un système de classification de réclamations financières.

Votre tâche consiste à attribuer UNE SEULE catégorie à chaque réclamation.

Voici des exemples de réclamation correctement classées:
{examples_text}
"""


In [334]:
from metrics import (
    calculate_metrics,
    print_metrics,
    report,
)

name = "test 20 avec exemples"
results_path = Path(name + ".pkl")
results = load_results(results_path)

if len(results) == 0:
    print("Démarre le test:", name)
    indices = X_test.index[:20]
    results = test(indices)
    save_results(results, results_path)

metrics = calculate_metrics(results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
report(results)


------------------------
test 20 avec exemples
------------------------
Accuracy : 55.00%
Precision (macro) : 25.31%
Recall (macro) : 26.39%
F1 (macro) : 25.82%
Precision (weighted) : 51.11%
Recall (weighted) : 55.00%
F1 (weighted) : 52.94%
Temps moyen (s) : 1.239 s
Temps médian (s) : 0.430 s
Temps P95 (s) : 1.337 s
------------------------
                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       0.00      0.00      0.00         2
                                                 Credit card or prepaid card       1.00      1.00      1.00         1
Credit reporting, credit repair services, or other personal consumer reports       0.50      0.50      0.50         6
                                                             Debt collection       0.78      0.88      0.82         8
                                                                 

## Test avec des exemples du RAG

# Evaluation

Evalue le système sur un échantillonnage plus important et affiche les metriques
(pour une utilisation avec une API limité le test peut être exécuté en plusieurs fois)

In [335]:
from metrics import (
    calculate_metrics,
    print_metrics,
    report,
)

results_path = Path("results.pkl")
results = load_results(results_path)

max_results = 50
name = "test final"
if len(results) < max_results:
    print("Démarre le test:", name)
    indices = X_test.index[len(results):max_results]
    additional_results = test(indices)
    results = pd.concat([results, additional_results], ignore_index=True)
    save_results(results, results_path)


In [336]:
results

,Index,Question,Réponse,Attendue,Correct,Temps (s)
0,192075,I generally let people walk over me you could ...,"Credit reporting, credit repair services, or o...","Credit reporting, credit repair services, or o...",True,0.683262
1,131727,MR. XXXX calls and tells me he is with the leg...,Debt collection,Debt collection,True,0.501690
2,455266,I am including my marriage license per your re...,Other financial service,"Credit reporting, credit repair services, or o...",False,0.427090
3,80575,Disputed with company on XX/XX/XXXX. The compa...,"Credit reporting, credit repair services, or o...",Debt collection,False,0.595353
4,551360,"On XXXX XXXX, XXXX, we turned-over our XXXX XX...",Vehicle loan or lease,"Payday loan, title loan, or personal loan",False,0.506092
5,694534,Transunion deleted XXXX XXXX and XXXX. I have ...,"Credit reporting, credit repair services, or o...","Credit reporting, credit repair services, or o...",True,0.658732
6,776888,I contacted CFPB two years ago about how Bayvi...,"Credit reporting, credit repair services, or o...",Mortgage,False,0.382551
7,192053,We have received multiple calls from Commerica...,Debt collection,Debt collection,True,0.606573
8,721113,XXXX alleged that I owe them {$76.00}. for a p...,Debt collection,Debt collection,True,0.325626
9,49478,To whom it may concern On XX/XX/XXXX Radius Gl...,Debt collection,"Credit reporting, credit repair services, or o...",False,0.418645


In [337]:
metrics = calculate_metrics(results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
report(results)


------------------------
test final
------------------------
Accuracy : 66.00%
Precision (macro) : 45.56%
Recall (macro) : 41.69%
F1 (macro) : 40.92%
Precision (weighted) : 78.44%
Recall (weighted) : 66.00%
F1 (weighted) : 69.20%
Temps moyen (s) : 0.474 s
Temps médian (s) : 0.457 s
Temps P95 (s) : 0.672 s
------------------------
                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       1.00      0.40      0.57         5
                                                 Credit card or prepaid card       0.60      0.60      0.60         5
Credit reporting, credit repair services, or other personal consumer reports       0.56      0.77      0.65        13
                                                             Debt collection       0.90      0.60      0.72        15
                          Money transfer, virtual currency, or money service